# Transformer From Scratch

Based on Neel Nanda's GPT-2 From Scratch

Adapted to use only HuggingFace and PyTorch, no other dependencies (Neel's EasyTransformers is great, but incompatible with some versions of )
Training vs inference?

Why GPT-2?

TODO add extended desc

TODO put in git repo, organize, publish. see how they fit together

# Setup, Utilities, etc

Markdown Syntax
$$
\begin{bmatrix}
a & b \\
c & d
\end{bmatrix}
$$

In [73]:
%pip install transformers
%pip install fancy_einsum
%pip install einops


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [74]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
import torch
import torch.nn as nn
import numpy as np
import math
import tqdm.auto as tqdm

In [75]:
#Config object to set params
@dataclass
class Config:
    d_model: int = 768 # size of residual string
    debug: bool = True
    layer_norm_epsilon: float = 1e-5 # added in layernorm to prevent division by 0
    d_vocab: int = 50257
    init_range: float = 0.02
    
cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_epsilon=1e-05, d_vocab=50257, init_range=0.02)


## Reference Model

Major adaptations:
- `model(x)` returns a `CausalLMOutputWithCrossAttentions`, which is a named tuple with `.logits`, `.past_key_values`, `.hidden_states`, `.attentions` - not just logits
- `run_with_cache` gives entries even for sub modules, use `c_attn` for concatenated QKV matrix `[B, T, 3*d_model]` before split
- Attention returns a tuple `(attn_output, present_key_value, attn_weights)`, so `output[0]` is the post-attention residual contribution that gets added back to the stream

In [76]:
# We don't use Neel's EasyTransformers since dependencies have changed
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)
device = "cpu" # hard code, some operations not yet supported on mps

model_name = "gpt2"  # this is gpt2-small

reference_gpt2 = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

reference_gpt2.eval()

mps


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8949.53it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [77]:
print(reference_gpt2)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [78]:
# helpers

def run_with_cache(model, tokens):
    cache = {}
    hooks = []
    
    def make_hook(name):
        def hook(module, input, output):
            if isinstance(output, torch.Tensor):
                cache[name] = output.detach()
            elif isinstance(output, tuple) and isinstance(output[0], torch.Tensor):
                cache[name] = output[0].detach()
        return hook
    
    for name, module in model.named_modules():
        hooks.append(module.register_forward_hook(make_hook(name)))
        
    with torch.no_grad():
        out = model(tokens)
        
    for h in hooks:
        h.remove()
        
    return out.logits, cache

Key:
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 3072 (4 * d_model)
d_head = 64 (d_model / n_heads)

In [79]:
# text to tokens
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = tokenizer.encode(reference_text, return_tensors="pt").to(device)
print(tokens)
print(tokens.shape) # [batch, position]
print(tokenizer.convert_ids_to_tokens(tokens[0].tolist()))

tensor([[   40,   716,   281,  4998,  1960,   382, 19741,    11,   875, 12342,
            12,  8807,    11,   402, 11571,    12,    17,  3918, 47385,    13,
          1881,  1110,   314,   481,  7074,  1692,  1241,  4430,   290,  1011,
           625,   262,   995,     0]])
torch.Size([1, 34])
['I', 'Ġam', 'Ġan', 'Ġamazing', 'Ġaut', 'ore', 'gressive', ',', 'Ġdec', 'oder', '-', 'only', ',', 'ĠG', 'PT', '-', '2', 'Ġstyle', 'Ġtransformer', '.', 'ĠOne', 'Ġday', 'ĠI', 'Ġwill', 'Ġexceed', 'Ġhuman', 'Ġlevel', 'Ġintelligence', 'Ġand', 'Ġtake', 'Ġover', 'Ġthe', 'Ġworld', '!']


In [80]:
# tokens to logits
logits, cache = run_with_cache(reference_gpt2, tokens)
print(logits.shape) # [batch, position]

torch.Size([1, 34, 50257])


In [81]:
# logits to distribution
log_probs = logits.log_softmax(dim=-1)
probs = logits.softmax(dim=-1)
print(log_probs.shape)
print(probs.shape)

torch.Size([1, 34, 50257])
torch.Size([1, 34, 50257])


In [82]:
# distribution to token
next_token = logits[0, -1].argmax(dim=-1) # -1 for the last token
print(next_token)
print(tokenizer.decode([next_token.item()]))

tensor(314)
 I


In [83]:
# repeat!
next_tokens = torch.cat(
    [tokens, next_token.reshape(1,1)], dim=-1)

with torch.no_grad():
    new_output = reference_gpt2(next_tokens)
    
new_logits = new_output.logits # [batch, position, vocab_size]

print("New input (tokens):", next_tokens)
print(next_tokens.shape)
print("New input (text):", tokenizer.decode(next_tokens[0].tolist()))

print(new_logits.shape)

predicted_token_id = new_logits[0, -1].argmax(-1)
print("Predicted next token:", tokenizer.decode([predicted_token_id.item()]))

New input (tokens): tensor([[   40,   716,   281,  4998,  1960,   382, 19741,    11,   875, 12342,
            12,  8807,    11,   402, 11571,    12,    17,  3918, 47385,    13,
          1881,  1110,   314,   481,  7074,  1692,  1241,  4430,   290,  1011,
           625,   262,   995,     0,   314]])
torch.Size([1, 35])
New input (text): I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world! I
torch.Size([1, 35, 50257])
Predicted next token:  am


In [84]:
# Activation Shapes
for activation_name, activation in cache.items():
    if not hasattr(activation, 'shape'):
        continue
    is_block_0 = "transformer.h.0" in activation_name
    is_top_level = "transformer.h" not in activation_name and activation_name != ""
    if is_block_0 or is_top_level: # only first block
        print(activation_name, tuple(activation.shape))

transformer.wte (1, 34, 768)
transformer.wpe (1, 34, 768)
transformer.drop (1, 34, 768)
transformer.h.0.ln_1 (1, 34, 768)
transformer.h.0.attn.c_attn (1, 34, 2304)
transformer.h.0.attn.c_proj (1, 34, 768)
transformer.h.0.attn.resid_dropout (1, 34, 768)
transformer.h.0.attn (1, 34, 768)
transformer.h.0.ln_2 (1, 34, 768)
transformer.h.0.mlp.c_fc (1, 34, 3072)
transformer.h.0.mlp.act (1, 34, 3072)
transformer.h.0.mlp.c_proj (1, 34, 768)
transformer.h.0.mlp.dropout (1, 34, 768)
transformer.h.0.mlp (1, 34, 768)
transformer.h.0 (1, 34, 768)
transformer.ln_f (1, 34, 768)
lm_head (1, 34, 50257)


## Test

In [85]:
# get_corner returns top-left corner of last two dimensions 
def get_corner(tensor, n=3):
    return tensor[..., :n, :n]

In [86]:
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg)
    random_input = torch.randn(shape) # standard gaussian
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print(get_corner(output))
    return output

def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg)
    random_input = torch.randint(low=0, high=cfg.d_vocab, size=shape) # standard gaussian
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print(get_corner(output))
    return output

def load_gpt2_test(cls, gpt2_layer, input_name, gpt2_layer_output):
    cfg = Config(debug=True)
    layer = cls(cfg)
    layer.load_state_dict(gpt2_layer.state_dict())
    if isinstance(input_name, str):
        reference_input = cache[input_name]
    else:
        reference_input = input_name
    print("Input shape:", reference_input.shape)
    output = layer(reference_input)
    print("Output shape:", output.shape)
    print("Reference output shape:", gpt2_layer_output.shape)
    
    comparison = torch.isclose(output, gpt2_layer_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of values are correct")
    return output

# define get_corner


# LayerNorm

- Applied at the start of each layer (MLP, attention)
- Normalizing function, taking an input vector and transforming it to have mean = 0 and variance = 1
- Also applies some non-linear scaling with the learned weights
- Neel describes this as an area where pretty cool math takes place - elementwise scaling and translation 

In [87]:
class LayerNorm(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w = nn.Parameter(torch.ones(cfg.d_model)) # norm by default = 1
        self.b = nn.Parameter(torch.zeros(cfg.d_model)) # mean by default = 0
        
    def forward(self, residual):
        # residual: [batch, position, d_model]
        if cfg.debug: print("Residual:", residual.shape)
        # einops.reduce(tensor, pattern, reduction, optional axes length) -> tensor
        residual = residual - einops.reduce(residual, "batch position d_model -> batch position 1", "mean")
        # calculate variance and square root, then add epsilon to ensure non-zero
        #   variance: sum of squares / n
        scale = (einops.reduce(residual.pow(2), "batch position d_model -> batch position 1", "mean") + cfg.layer_norm_epsilon).sqrt()
        residual = residual / scale
        normalized = residual * self.w + self.b
        if cfg.debug: print("Normalized:", residual.shape)
        return normalized

In [88]:
rand_float_test(LayerNorm, [2, 4, 768])
load_gpt2_test(LayerNorm, reference_gpt2.transformer.h.ln_2, LayerNorm, tokens)

Input shape: torch.Size([2, 4, 768])
Residual: torch.Size([2, 4, 768])
Normalized: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])
tensor([[[-1.2228, -1.2494,  1.2814],
         [ 0.5219,  0.4140,  1.1249],
         [-2.1482,  0.9436,  0.8858]],

        [[ 0.2807,  1.0379, -0.7704],
         [-0.6490,  0.1138,  0.2678],
         [ 1.9910,  0.3505, -0.7480]]], grad_fn=<SliceBackward0>)


AttributeError: 'ModuleList' object has no attribute 'ln_2'

# Embedding

- A lookup table from input token to vector
- Note that the process of tokenizing (raw input language to tokens) uses Byte-Pair Encoding (shoutout CS240E at Waterloo!). We start with only having tokens for individual letters (ie. `a=1, b=2, c=3` - a bit of an oversimplication), and merging the most common groups of sequences to create a vocabulary of short strings (ie. the pair `a` + `b` occurs often, so we create in our dictionary `ab=4`), some of which are complete words and others are not. This forms our vocabulary of tokens, where each string corresponds to an integer.
- We take an input `token_1, token_2, token_3, ...` and convert it into a tensor (matrix) via matrix multiplication with one-hot embedded vectors
- `d_vocab x d_model`

`token` (an integer) $\cdot \begin{bmatrix} 0 \\ 0 \\ 0 \\ ... \\ 1 \\ ... \\ 0 \end{bmatrix} $



In [ ]:
class Embed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(torch.randn(cfg.d_vocab, cfg.d_model)) # why 0.02? why not?
        nn.init.normal_(self.W_E, std=self.cfg.init_range)
    
    def forward(self, tokens):
        # tokens: [batch, position]
        if cfg.debug: print("Tokens:", tokens.shape)
        embed = self.W_E[tokens, :] # [batch, position, d_model]
        if cfg.debug: print("Embeddings:", embed.shape)
        return embed

In [ ]:
rand_int_test(Embed, [2, 4])
load_gpt2_test(Embed, reference_gpt2.embed, tokens)

Input shape: torch.Size([2, 4])
Tokens: torch.Size([2, 4])
Embeddings: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])
tensor([[[ 0.0102, -0.0226,  0.0029],
         [-0.0011, -0.0177,  0.0155],
         [-0.0384, -0.0169,  0.0123]],

        [[ 0.0077,  0.0254,  0.0049],
         [ 0.0138, -0.0013, -0.0234],
         [-0.0054,  0.0311,  0.0176]]], grad_fn=<SliceBackward0>)


AttributeError: 'GPT2LMHeadModel' object has no attribute 'embed'

# Postitional Embedding
- We add information about the position of each token. 
- TODO check: Note that tokens that are close together should have a greater influence on one another - intuitively, in the sentence "the dancer leaped across the stage, and the lively crowd applauded", "lively" impacts the meaning of "crowd" a lot more than "dancer".

# Attention
- This is the only step that moves information from a prior position in the sequence to the current one
- We parallelize this - we do this for each token for prefix token strings
- n heads etc

# MLP

# Unembedding

- Transforms the final vector into a tensor of logits using a softmax function
$ softmax(x_i) = \frac{e^{x_i}}{\sum e^{x_j}} $
- This output is a tensor of logits, where we have one vector of size $d_{vocab}$ for each input token. 

$$
\begin{bmatrix}
a_0 & b_0 & ...\\
a_1 & b_1 & ...\\
... \\
a_{d} & b_d & ...
\end{bmatrix}
$$